# GCN Traffic Flow Prediction
Graph Convolutional Network on PEMS-BAY dataset (325 sensors, 52116 timesteps)

## 1. Imports

In [ ]:
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Load Data

In [ ]:
with h5py.File('Data/pems-bay.h5', 'r') as f:
    speed = f['speed']['block0_values'][:]   # (52116, 325)
    sensor_ids = f['speed']['axis0'][:]      # (325,)

meta = pd.read_csv('Data/PEMS-BAY-META.csv')
meta = meta.sort_values('sensor_id').reset_index(drop=True)

print('Speed shape:', speed.shape)
print('Missing values:', np.isnan(speed).sum())

## 3. Graph Construction
Build adjacency matrix using thresholded Gaussian kernel on pairwise GPS distances.

$$W_{ij} = \exp\left(-\frac{d_{ij}^2}{\sigma^2}\right) \text{ if } d_{ij} \leq \epsilon, \text{ else } 0$$

In [ ]:
from sklearn.metrics.pairwise import haversine_distances
from math import radians

coords = meta[['Latitude', 'Longitude']].values
coords_rad = np.radians(coords)

# Haversine distances in km
dist_matrix = haversine_distances(coords_rad) * 6371

sigma = dist_matrix.std()
epsilon = 0.5  # km threshold — only connect sensors within 0.5 km

adj = np.exp(-dist_matrix**2 / sigma**2)
adj[dist_matrix > epsilon] = 0
np.fill_diagonal(adj, 0)  # no self-loops (added in GCN layer)

print('Adjacency matrix shape:', adj.shape)
print('Non-zero edges:', np.count_nonzero(adj))
print('Avg degree:', np.count_nonzero(adj) / len(adj))

In [ ]:
def normalize_adj(adj):
    """Symmetric normalization: D^{-1/2} A D^{-1/2} with added self-loops."""
    A = adj + np.eye(adj.shape[0])  # add self-loops
    D = np.diag(A.sum(axis=1) ** -0.5)
    return D @ A @ D

adj_norm = normalize_adj(adj)
adj_tensor = torch.FloatTensor(adj_norm).to(device)
print('Normalized adjacency ready:', adj_tensor.shape)

## 4. Data Preprocessing

In [ ]:
scaler = StandardScaler()
speed_scaled = scaler.fit_transform(speed)  # (52116, 325)

SEQ_LEN = 12
X, y = [], []
for i in range(len(speed_scaled) - SEQ_LEN):
    X.append(speed_scaled[i:i + SEQ_LEN])
    y.append(speed_scaled[i + SEQ_LEN])

X = np.array(X, dtype=np.float32)  # (N, 12, 325)
y = np.array(y, dtype=np.float32)  # (N, 325)

train_size = int(len(X) * 0.70)
val_size   = int(len(X) * 0.15)

X_train, y_train = X[:train_size], y[:train_size]
X_val,   y_val   = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test,  y_test  = X[train_size+val_size:], y[train_size+val_size:]

print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)

In [ ]:
class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 64
train_loader = DataLoader(TrafficDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TrafficDataset(X_val,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(TrafficDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

## 5. GCN Model

Architecture:
```
Input: (batch, seq_len=12, nodes=325)
  → For each timestep: GCN layer (325 → hidden)
  → Mean pool over time
  → Linear (hidden → 1) per node
Output: (batch, 325)
```

In [ ]:
class GCNLayer(nn.Module):
    """Single GCN layer: H' = σ(A_norm * H * W)"""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, adj):
        # x: (batch, nodes, features)
        # adj: (nodes, nodes)
        return F.relu(adj @ self.W(x))


class GCN(nn.Module):
    def __init__(self, num_nodes, seq_len, hidden=64):
        super().__init__()
        self.gcn1 = GCNLayer(seq_len, hidden)
        self.gcn2 = GCNLayer(hidden, hidden)
        self.out  = nn.Linear(hidden, 1)

    def forward(self, x, adj):
        # x: (batch, seq_len, nodes) → transpose to (batch, nodes, seq_len)
        x = x.permute(0, 2, 1)
        x = self.gcn1(x, adj)   # (batch, nodes, hidden)
        x = self.gcn2(x, adj)   # (batch, nodes, hidden)
        x = self.out(x).squeeze(-1)  # (batch, nodes)
        return x


NUM_NODES = 325
model = GCN(num_nodes=NUM_NODES, seq_len=SEQ_LEN, hidden=64).to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 6. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

EPOCHS = 30
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb, adj_tensor)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model(xb, adj_tensor), yb).item()

    train_losses.append(total_loss / len(train_loader))
    val_losses.append(val_loss / len(val_loader))

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}')

In [ ]:
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('GCN Training Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluation

In [ ]:
model.eval()
preds, actuals = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb, adj_tensor).cpu().numpy()
        preds.append(pred)
        actuals.append(yb.numpy())

preds   = np.concatenate(preds)    # (N_test, 325) — scaled
actuals = np.concatenate(actuals)  # (N_test, 325) — scaled

# Inverse transform to original mph scale
preds_mph   = scaler.inverse_transform(preds)
actuals_mph = scaler.inverse_transform(actuals)

In [ ]:
mae  = mean_absolute_error(actuals_mph, preds_mph)
rmse = np.sqrt(np.mean((actuals_mph - preds_mph) ** 2))
# MAPE — avoid division by zero
mask = actuals_mph != 0
mape = np.mean(np.abs((actuals_mph[mask] - preds_mph[mask]) / actuals_mph[mask])) * 100

print(f'MAE  : {mae:.4f} mph')
print(f'RMSE : {rmse:.4f} mph')
print(f'MAPE : {mape:.2f} %')

In [ ]:
# Visualise predictions vs actuals for sensor 0 over first 200 test steps
plt.figure(figsize=(12, 4))
plt.plot(actuals_mph[:200, 0], label='Actual')
plt.plot(preds_mph[:200, 0],   label='Predicted', alpha=0.8)
plt.xlabel('Time step')
plt.ylabel('Speed (mph)')
plt.title('GCN Prediction vs Actual — Sensor 0')
plt.legend()
plt.tight_layout()
plt.show()